In [3]:
using LinearAlgebra
using Plots

# --- パラメータ設定 ---
const c = 137.036      # 光速 (原子単位系 a.u.)
const m = 1.0          # 電子の質量 (a.u.)
const ħ = 1.0          # プランク定数 (a.u.)

# 空間グリッドの設定
const L = 2.0          # 空間の範囲 (-L ~ L)
const N = 400          # グリッド点数
const x = range(-L, L, length=N)
const dx = x[2] - x[1]

# 時間刻みの設定 (クーラン条件 c*dt < dx を満たす必要がある)
const dt = 0.1 * dx / c 
const T_max = 0.05     # シミュレーションの終了時間
const steps = floor(Int, T_max / dt)

# --- 行列の定義 (1+1次元ディラック方程式) ---
# 1次元ではパウリ行列 σx, σz を α, β として使用できます
const σx = [0.0 1.0; 1.0 0.0]
const σz = [1.0 0.0; 0.0 -1.0]
const α = σx
const β = σz

# --- 初期条件: ガウス波束 ---
# 波束の中心位置と運動量
x0 = -0.5
k0 = 100.0  # 初期運動量
width = 0.1 # 波束の幅

# 初期波動関数 ψ (2成分スピノル: [ψ1, ψ2])
# N個の点それぞれに2成分ベクトルを持つため、(2, N) の配列にします
ψ = zeros(ComplexF64, 2, N)

# ガウス分布で初期化
norm_factor = (1.0 / (π * width^2))^(0.25)
for i in 1:N
    # 空間部分
    envelope = norm_factor * exp(-(x[i] - x0)^2 / (2 * width^2))
    phase = exp(im * k0 * x[i])
    
    # スピノル部分（ここでは上向きスピン成分を主とする）
    # 運動量を持つ自由粒子の解に近い構成比にするのが一般的ですが、
    # 簡単のため (1, 0) 成分に入れます。
    spinor = [1.0, 0.0] 
    
    ψ[:, i] = envelope * phase * spinor
end

# 正規化 (全確率を1にする)
total_prob = sum(sum(abs2.(ψ), dims=1)) * dx
ψ ./= sqrt(total_prob)

# --- 時間発展の計算 (ルンゲ・クッタ法: RK4) ---

# ディラック方程式のハミルトニアン作用 Hψ
# H = -i ħ c α ∂x + m c^2 β
function apply_hamiltonian(psi, dx, c, m, hbar, α, β)
    rows, cols = size(psi)
    dpsi = zeros(ComplexF64, rows, cols)
    
    # 空間微分の計算 (中心差分法)
    # 境界条件は0 (ディリクレ) とする
    for i in 2:cols-1
        d_dx = (psi[:, i+1] - psi[:, i-1]) / (2 * dx)
        
        # Hψ = -i c α (∂ψ/∂x) + m c^2 β ψ
        term1 = -im * c * ħ * (α * d_dx)
        term2 = m * c^2 * (β * psi[:, i])
        
        dpsi[:, i] = (term1 + term2) / (im * ħ) # i hbar dpsi/dt = H psi -> dpsi/dt = (1/ i hbar) H psi
    end
    return dpsi
end

# アニメーションの作成
anim = @animate for t in 1:100:steps
    global ψ
    
    # RK4 ステップ
    # ループ内で複数回回して時間進行を速めることも可能
    for _ in 1:10 # 表示間隔の間に10ステップ進める
        k1 = apply_hamiltonian(ψ, dx, c, m, ħ, α, β)
        k2 = apply_hamiltonian(ψ + 0.5 * dt * k1, dx, c, m, ħ, α, β)
        k3 = apply_hamiltonian(ψ + 0.5 * dt * k2, dx, c, m, ħ, α, β)
        k4 = apply_hamiltonian(ψ + dt * k3, dx, c, m, ħ, α, β)
        
        ψ += (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)
    end
    
    # 確率密度 |ψ1|^2 + |ψ2|^2
    prob_density = vec(sum(abs2.(ψ), dims=1))
    
    # プロット
    plot(x, prob_density, 
         title = "Dirac Equation Simulation (1D)",
         label = "|ψ|² (Probability Density)",
         xlabel = "Position (x)",
         ylabel = "Probability",
         ylims = (0, 3.0),
         lw = 2,
         color = :blue,
         fill = (0, 0.2, :blue)
    )
    
    # 実部と虚部、あるいはスピノル成分ごとの密度を見たい場合は以下を追加
    plot!(x, abs2.(ψ[1, :]), label="|ψ_up|²", color=:red, linestyle=:dash)
    plot!(x, abs2.(ψ[2, :]), label="|ψ_down|²", color=:green, linestyle=:dash)
end

# 保存
gif(anim, "dirac_1d.gif", fps = 15)
println("アニメーションが 'dirac_1d.gif' として保存されました。")

アニメーションが 'dirac_1d.gif' として保存されました。


[ Info: Saved animation to /home/jovyan/work/dirac_1d.gif
